# Keyword Grouping Pipeline

Groups similar/duplicate research keywords from `aggregate.json` using:
- **Step 1** — Rule-based normalization (instant, no model needed)
- **Step 2** — Semantic grouping via Ollama (local LLM)
- **Step 3** — Merge and save output

**Requirements:** `ollama serve` must be running and the model pulled.

## Cell 1 — Configuration

In [1]:
INPUT_FILE   = "aggregate.json"
OUTPUT_FILE  = "keywords_grouped.json"
LOG_FILE     = "grouping_log.txt"

OLLAMA_MODEL        = "llama3.2"
OLLAMA_URL          = "http://localhost:11434/api/generate"
SEMANTIC_BATCH_SIZE = 150
RETRY_DELAY         = 3

print(f"Model: {OLLAMA_MODEL} | Batch size: {SEMANTIC_BATCH_SIZE}")

Model: llama3.2 | Batch size: 150


## Cell 2 — Imports and Helpers

In [2]:
import json, re, time, unicodedata, urllib.request, urllib.error
from collections import defaultdict
from tqdm.notebook import tqdm

ABBREV_MAP = {
    "co2": "carbon dioxide", "h2o": "water", "h2s": "hydrogen sulfide",
    "ch4": "methane", "n2": "nitrogen", "o2": "oxygen", "h2": "hydrogen",
    "nox": "nitrogen oxides", "sox": "sulfur oxides",
    "vle": "vapor-liquid equilibrium", "lle": "liquid-liquid equilibrium",
    "sle": "solid-liquid equilibrium",
    "md": "molecular dynamics", "mc": "monte carlo",
    "dft": "density functional theory",
    "ml": "machine learning", "dl": "deep learning",
    "nn": "neural network", "ai": "artificial intelligence",
    "cfd": "computational fluid dynamics",
    "pde": "partial differential equation", "ode": "ordinary differential equation",
    "il": "ionic liquid", "ils": "ionic liquid",
    "mea": "monoethanolamine", "mdea": "methyldiethanolamine",
    "pz": "piperazine", "dea": "diethanolamine",
    "ccs": "carbon capture and storage",
    "ccus": "carbon capture utilization and storage",
    "tga": "thermogravimetric analysis",
    "dsc": "differential scanning calorimetry",
    "nmr": "nuclear magnetic resonance",
    "ftir": "fourier transform infrared spectroscopy",
    "gc": "gas chromatography", "hplc": "high performance liquid chromatography",
    "pvt": "pressure volume temperature", "eos": "equation of state",
    "saft": "statistical associating fluid theory",
    "pr": "peng-robinson", "srk": "soave-redlich-kwong",
    "mof": "metal-organic framework", "cof": "covalent organic framework",
    "htc": "hydrothermal carbonization", "htl": "hydrothermal liquefaction",
    "cstr": "continuous stirred tank reactor", "pfr": "plug flow reactor",
    "mpc": "model predictive control", "pid": "proportional integral derivative",
    "lca": "life cycle assessment", "tac": "total annualized cost",
    "capex": "capital expenditure", "opex": "operational expenditure",
    "ro": "reverse osmosis", "mf": "microfiltration",
    "uf": "ultrafiltration", "nf": "nanofiltration",
    "ed": "electrodialysis", "ghg": "greenhouse gas",
}

def normalize(text):
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[\u2010-\u2015\u2212\ufe58\ufe63\uff0d]", "-", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"-+", "-", text)
    return text

def depluralize(text):
    if text.endswith("ies") and len(text) > 4: return text[:-3] + "y"
    if text.endswith("ses") and len(text) > 4: return text[:-2]
    if text.endswith("s") and not text.endswith("ss") and len(text) > 3: return text[:-1]
    return text

def rule_based_canonical(kw):
    norm  = normalize(kw)
    words = [ABBREV_MAP.get(w, w) for w in norm.split()]
    if words: words[-1] = depluralize(words[-1])
    return " ".join(words)

def merge_keyword_data(base, extra):
    base["total_count"] = base.get("total_count", 0) + extra.get("total_count", 0)
    for year, cnt in extra.get("years", {}).items():
        base.setdefault("years", {})[year] = base["years"].get(year, 0) + cnt
    for country, cnt in extra.get("countries", {}).items():
        base.setdefault("countries", {})[country] = base["countries"].get(country, 0) + cnt
    for author, cnt in extra.get("authors", {}).items():
        base.setdefault("authors", {})[author] = base["authors"].get(author, 0) + cnt
    for year, papers in extra.get("papers_by_year", {}).items():
        existing = base.setdefault("papers_by_year", {}).setdefault(year, [])
        seen = {(p["title"], p.get("source_file", "")) for p in existing}
        for p in papers:
            key = (p["title"], p.get("source_file", ""))
            if key not in seen:
                existing.append(p)
                seen.add(key)
    return base

def call_ollama(prompt, retries=3):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 2048}
    }).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=300) as resp:
                return json.loads(resp.read()).get("response", "")
        except urllib.error.URLError as e:
            if attempt < retries - 1:
                time.sleep(RETRY_DELAY)
            else:
                raise RuntimeError(f"Ollama unreachable: {e}")
        except Exception:
            if attempt < retries - 1:
                time.sleep(RETRY_DELAY)
            else:
                raise

print("Helpers loaded.")

Helpers loaded.


## Cell 3 — Load Data

In [3]:
with open(INPUT_FILE, encoding="utf-8") as f:
    data = json.load(f)

print(f"Keywords loaded : {len(data):,}")
print(f"Sample          : {list(data.keys())[:5]}")

Keywords loaded : 19,592
Sample          : ['vapor-liquid equilibrium', 'acid gas absorption', 'mdea‑piperazine blends', 'opls‑aa force field', 'accurate point charges']


## Cell 4 — Step 1: Rule-Based Grouping

In [4]:
norm_to_originals = defaultdict(list)
for kw in tqdm(data.keys(), desc="Normalizing"):
    norm_to_originals[rule_based_canonical(kw)].append(kw)

rule_groups = {}
ungrouped   = []

for norm, originals in norm_to_originals.items():
    if len(originals) == 1:
        ungrouped.append(originals[0])
    else:
        canonical = max(originals, key=lambda k: data[k].get("total_count", 0))
        rule_groups[canonical] = [k for k in originals if k != canonical]

merged_count = sum(len(v) for v in rule_groups.values())
print(f"Rule groups     : {len(rule_groups):,}")
print(f"Keywords merged : {merged_count:,}")
print(f"Ungrouped left  : {len(ungrouped):,}")
print()
print("Sample groups:")
for canon, aliases in list(rule_groups.items())[:5]:
    print(f"  {canon!r:40s} <- {aliases}")

Normalizing:   0%|          | 0/19592 [00:00<?, ?it/s]

Rule groups     : 392
Keywords merged : 404
Ungrouped left  : 18,796

Sample groups:
  'vapor-liquid equilibrium'               <- ['vapor‑liquid equilibrium']
  'redlich-kister correlation'             <- ['redlich‑kister correlation']
  'vibrating-tube densimetry'              <- ['vibrating‑tube densimetry']
  'computational fluid dynamics'           <- ['cfd']
  'cation exchange membrane'               <- ['cation exchange membranes']


## Cell 5 — Check Ollama

In [5]:
try:
    with urllib.request.urlopen("http://localhost:11434", timeout=5) as r:
        print(f"Ollama running  (HTTP {r.status})")
except Exception as e:
    print(f"Ollama NOT reachable: {e}")
    print(f"  -> open a terminal and run: ollama serve")
    print(f"  -> then pull the model:     ollama pull {OLLAMA_MODEL}")

Ollama running  (HTTP 200)


## Cell 6 — Step 2: Semantic Grouping

Sends batches of keywords to your local Ollama model.
With RTX 3050 + llama3.2 expect ~30–45 s/batch.
Missed batches (parse errors / timeouts) are saved to `missed_keywords.txt` instead of dropped.

In [6]:
SEMANTIC_PROMPT = (
    "You are a research keyword deduplication expert for chemical engineering "
    "and related sciences.\n\n"
    "Below is a numbered list of research keywords. Identify groups that refer "
    "to the SAME or very similar concept. Be conservative — only group keywords "
    "that clearly mean the same thing.\n\n"
    "Keywords:\n{keyword_list}\n\n"
    "Return ONLY a JSON array where each element has keys \"canonical\" and \"members\".\n"
    "- \"members\" must include the canonical keyword itself\n"
    "- Only include groups with 2+ members\n"
    "- Do NOT return list of lists\n"
    "- Do NOT explain anything"
)

MISSED_FILE = "missed_keywords.txt"

semantic_groups = []
missed_keywords = []
failed_batches  = []
ok_batches      = 0

total_batches = (len(ungrouped) + SEMANTIC_BATCH_SIZE - 1) // SEMANTIC_BATCH_SIZE

print(f"Keywords  : {len(ungrouped):,}")
print(f"Model     : {OLLAMA_MODEL}")
print(f"Batches   : {total_batches}  (size {SEMANTIC_BATCH_SIZE})")
print()

pbar = tqdm(total=total_batches, unit="batch",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}  [{elapsed}<{remaining}, {rate_fmt}]  {postfix}")
pbar.set_postfix(ok=0, fail=0, groups=0)

for batch_num in range(total_batches):
    start = batch_num * SEMANTIC_BATCH_SIZE
    batch = ungrouped[start : start + SEMANTIC_BATCH_SIZE]

    kw_list = "\n".join(f"{i+1}. {kw}" for i, kw in enumerate(batch))
    prompt  = SEMANTIC_PROMPT.format(keyword_list=kw_list)

    groups_this_batch = 0
    t0 = time.time()

    try:
        raw     = call_ollama(prompt)
        elapsed = time.time() - t0

        match = re.search(r"\[.*\]", raw, re.DOTALL)
        text  = match.group(0) if match else raw.strip()
        text  = re.sub(r"^```[a-z]*\n?", "", text)
        text  = re.sub(r"\n?```$", "", text).strip()

        if text and text != "[]":
            try:
                groups_raw = json.loads(text)
            except json.JSONDecodeError:
                missed_keywords.extend(batch)
                failed_batches.append(batch_num + 1)
                pbar.set_postfix(ok=ok_batches, fail=len(failed_batches), groups=len(semantic_groups))
                pbar.update(1)
                time.sleep(0.2)
                continue

            batch_lower = {k.lower().strip(): k for k in batch}

            for g in groups_raw:
                if isinstance(g, list):
                    members, canonical = g, (g[0] if g else "")
                elif isinstance(g, dict):
                    members   = g.get("members", [])
                    canonical = g.get("canonical", "")
                else:
                    continue

                resolved = [batch_lower[str(m).lower().strip()]
                            for m in members if str(m).lower().strip() in batch_lower]

                if len(resolved) >= 2:
                    canon_orig = batch_lower.get(str(canonical).lower().strip()) or resolved[0]
                    semantic_groups.append({"canonical": canon_orig, "members": resolved})
                    groups_this_batch += 1

    except Exception as e:
        elapsed = time.time() - t0
        missed_keywords.extend(batch)
        failed_batches.append(batch_num + 1)
        tqdm.write(f"  Batch {batch_num+1:>3} failed ({elapsed:.1f}s): {e}")
        pbar.set_postfix(ok=ok_batches, fail=len(failed_batches), groups=len(semantic_groups))
        pbar.update(1)
        time.sleep(0.2)
        continue

    ok_batches += 1
    pbar.set_postfix(ok=ok_batches, fail=len(failed_batches), groups=len(semantic_groups))
    pbar.update(1)
    time.sleep(0.2)

pbar.close()

if missed_keywords:
    with open(MISSED_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(missed_keywords))
    print(f"\nMissed keywords ({len(missed_keywords):,}) saved to {MISSED_FILE}")

print(f"\nSemantic groups : {len(semantic_groups):,}")
print(f"Failed batches  : {len(failed_batches)}" + (f"  {failed_batches}" if failed_batches else ""))

Keywords  : 18,796
Model     : llama3.2
Batches   : 126  (size 150)



  0%|          | 0/126  [00:00<?, ?batch/s]  


Missed keywords (12,750) saved to missed_keywords.txt

Semantic groups : 554
Failed batches  : 85  [1, 4, 6, 9, 10, 11, 12, 13, 14, 15, 17, 18, 20, 21, 22, 23, 24, 27, 28, 30, 32, 34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 57, 58, 59, 60, 61, 62, 65, 66, 67, 68, 70, 71, 72, 75, 76, 77, 78, 79, 80, 81, 82, 86, 87, 88, 90, 91, 93, 94, 96, 97, 99, 100, 101, 104, 107, 108, 111, 112, 114, 117, 119, 122, 123, 124, 125]


## Cell 7 — Step 3: Merge and Build Output

In [7]:
all_merges = {}
for canonical, aliases in rule_groups.items():
    all_merges[canonical] = list(aliases)
for g in semantic_groups:
    canonical = g["canonical"]
    aliases   = [m for m in g["members"] if m != canonical]
    if canonical in all_merges:
        all_merges[canonical].extend(aliases)
    else:
        all_merges[canonical] = aliases

all_aliases = {alias for aliases in all_merges.values() for alias in aliases}

result = {}
for kw, kw_data in tqdm(data.items(), desc="Building output"):
    if kw in all_aliases:
        continue
    entry = json.loads(json.dumps(kw_data))
    if kw in all_merges:
        for alias in all_merges[kw]:
            if alias in data:
                entry = merge_keyword_data(entry, data[alias])
        entry["grouped_with"] = all_merges[kw]
    else:
        entry["grouped_with"] = []
    entry["canonical_keyword"] = kw
    result[kw] = entry

total_groups  = sum(1 for v in result.values() if v["grouped_with"])
total_aliases = sum(len(v["grouped_with"]) for v in result.values())

print(f"Original keywords : {len(data):,}")
print(f"Output keywords   : {len(result):,}")
print(f"Keywords merged   : {total_aliases:,}")
print(f"Groups formed     : {total_groups:,}")

Building output:   0%|          | 0/19592 [00:00<?, ?it/s]

Original keywords : 19,592
Output keywords   : 18,491
Keywords merged   : 1,030
Groups formed     : 803


## Cell 8 — Save Output

In [8]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"Saved: {OUTPUT_FILE}")

with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("KEYWORD GROUPING LOG\n" + "=" * 60 + "\n\n")
    f.write(f"Original keywords  : {len(data)}\n")
    f.write(f"Output keywords    : {len(result)}\n")
    f.write(f"Keywords merged    : {total_aliases}\n")
    f.write(f"Groups formed      : {total_groups}\n\n")
    f.write("RULE-BASED GROUPS\n" + "-" * 40 + "\n")
    for canon, aliases in sorted(rule_groups.items()):
        f.write(f"  [{canon}]  <-  {aliases}\n")
    f.write("\nSEMANTIC GROUPS\n" + "-" * 40 + "\n")
    for g in semantic_groups:
        aliases = [m for m in g["members"] if m != g["canonical"]]
        f.write(f"  [{g['canonical']}]  <-  {aliases}\n")
print(f"Saved: {LOG_FILE}")
print("Done.")

Saved: keywords_grouped.json
Saved: grouping_log.txt
Done.


## Cell 9 — Preview Results

In [9]:
grouped_entries = {k: v for k, v in result.items() if v["grouped_with"]}
print(f"Total grouped keywords: {len(grouped_entries):,}\n")

for canon, entry in list(grouped_entries.items())[:20]:
    print(f"  CANONICAL : {canon}")
    print(f"  MERGED    : {entry['grouped_with']}")
    print(f"  COUNT     : {entry['total_count']}")
    print()

Total grouped keywords: 803

  CANONICAL : vapor-liquid equilibrium
  MERGED    : ['vapor‑liquid equilibrium']
  COUNT     : 8

  CANONICAL : redlich-kister correlation
  MERGED    : ['redlich‑kister correlation']
  COUNT     : 2

  CANONICAL : vibrating-tube densimetry
  MERGED    : ['vibrating‑tube densimetry']
  COUNT     : 2

  CANONICAL : computational fluid dynamics
  MERGED    : ['cfd']
  COUNT     : 21

  CANONICAL : cation exchange membrane
  MERGED    : ['cation exchange membranes']
  COUNT     : 4

  CANONICAL : techno-economic analysis
  MERGED    : ['techno‑economic analysis']
  COUNT     : 58

  CANONICAL : porous media hydrate dynamics
  MERGED    : ['gas hydrate multiphase flow']
  COUNT     : 2

  CANONICAL : mixed-integer nonlinear programming
  MERGED    : ['mixed‑integer nonlinear programming']
  COUNT     : 8

  CANONICAL : compression integration
  MERGED    : ['heat exchanger network design']
  COUNT     : 2

  CANONICAL : pfas water contamination
  MERGED    : [

## Cell 10 — Add Custom Abbreviations (Optional)

In [10]:
EXTRA_ABBREVS = {
    # "rxn": "reaction",
}
ABBREV_MAP.update(EXTRA_ABBREVS)
print(f"ABBREV_MAP has {len(ABBREV_MAP)} entries. Re-run from Cell 4 to apply.")

ABBREV_MAP has 59 entries. Re-run from Cell 4 to apply.
